# Amazon US Marketplace — Data Cleaning

In [1]:
#installing pandas
!pip install pandas numpy sqlalchemy pymysql

Defaulting to user installation because normal site-packages is not writeable


# Import libraries

In [20]:
import pandas as pd
import numpy as np
print('Libraries loaded successfully!')

Libraries loaded successfully!


# Load the Excel file

In [23]:
import os
os.chdir(r"C:\Users\JANVIPRIYANSHI\OneDrive\Desktop\amazon2")
print(os.listdir())

['____amazon_us_marketplace_RAw___.csv']


In [27]:
df = pd.read_csv(r"C:\Users\JANVIPRIYANSHI\OneDrive\Desktop\amazon2\____amazon_us_marketplace_RAw___.csv")
print(df.shape)
df.head()

(3715, 8)


,Merchant_Category,Buyer_Category,Departments,Volume,Revenue,Transaction_Count,Average selling price,Date
0,Very Low Volume,Low Engaged,Other,570,24.55,281,2.03,24-11-2014 00:00
1,Medium Volume,Super Engaged,Sports And Outdoors,20733354,751838.97,139536,148.59,27-02-2014 00:00
2,Unverified Merchant,Guest,"Movies, Music And Games",-196,28.60,24,-8.15,13-01-2014 00:00
3,Very High Volume,One-And-Done,Clothing And Shoes,142136,4403.07,275,516.86,21-05-2014 00:00
4,Very Low Volume,Guest,Books,839179,30726.21,17538,47.85,06-05-2014 00:00


In [29]:
print('Rows and Columns:', df.shape)

Rows and Columns: (3715, 8)


# Check data types and nulls

In [32]:
df.dtypes

Merchant_Category         object
Buyer_Category            object
Departments               object
Volume                     int64
Revenue                  float64
Transaction_Count          int64
Average selling price    float64
Date                      object
dtype: object

In [60]:
df['Date'] = pd.to_datetime(df['Date'], dayfirst=True)

In [34]:
df.isnull().sum()

Merchant_Category        0
Buyer_Category           0
Departments              0
Volume                   0
Revenue                  0
Transaction_Count        0
Average selling price    0
Date                     0
dtype: int64

In [40]:
df.duplicated().sum()

0

# Rename column

In [43]:
df = df.rename(columns={'Average selling price': 'Avg_Selling_Price'})

In [45]:
df.head()

,Merchant_Category,Buyer_Category,Departments,Volume,Revenue,Transaction_Count,Avg_Selling_Price,Date
0,Very Low Volume,Low Engaged,Other,570,24.55,281,2.03,24-11-2014 00:00
1,Medium Volume,Super Engaged,Sports And Outdoors,20733354,751838.97,139536,148.59,27-02-2014 00:00
2,Unverified Merchant,Guest,"Movies, Music And Games",-196,28.60,24,-8.15,13-01-2014 00:00
3,Very High Volume,One-And-Done,Clothing And Shoes,142136,4403.07,275,516.86,21-05-2014 00:00
4,Very Low Volume,Guest,Books,839179,30726.21,17538,47.85,06-05-2014 00:00


# Handle negative Volume rows

In [56]:
(df['Volume'] < 0).sum()
df[df['Volume'] < 0][['Date','Merchant_Category', 'Departments', 'Volume']]

,Date,Merchant_Category,Departments,Volume
2,13-01-2014 00:00,Unverified Merchant,"Movies, Music And Games",-196
174,20-03-2014 00:00,Inactive Merchant,Sports And Outdoors,-1673
428,07-08-2014 00:00,Inactive Merchant,Sports And Outdoors,-2089
437,09-07-2014 00:00,Inactive Merchant,Home And Garden,-16
552,12-11-2014 00:00,Unverified Merchant,Digital Games And Software,-125
...,...,...,...,...
3710,03-06-2017 00:00,Inactive Merchant,Sports And Outdoors,-5828
3711,23-03-2015 00:00,Low Volume,Other,-8847
3712,08-03-2016 00:00,Non Merchant,Automotive,-2839045
3713,06-01-2015 00:00,Medium Volume,Digital Games And Software,-111598216


In [66]:
# Add a flag column to label returns
df['Volume_Flag'] = 'Normal'
df.loc[df['Volume'] < 0, 'Volume_Flag'] = 'Return'

In [68]:
df['Volume_Flag'].value_counts()

Volume_Flag
Normal    3283
Return     432
Name: count, dtype: int64

In [73]:
df['Volume_Flag'].unique()

array(['Normal', 'Return'], dtype=object)

In [75]:
# Split into clean data and returns
df_clean   = df[df['Volume'] >= 0].copy()   # main dataset for analysis
df_returns = df[df['Volume'] < 0].copy()    # returns — kept separately

print('Clean rows :', len(df_clean))
print('Return rows:', len(df_returns))

Clean rows : 3283
Return rows: 432


In [80]:
df_returns

,Merchant_Category,Buyer_Category,Departments,Volume,Revenue,Transaction_Count,Avg_Selling_Price,Date,Volume_Flag
2,Unverified Merchant,Guest,"Movies, Music And Games",-196,28.60,24,-8.15,2014-01-13,Return
174,Inactive Merchant,Getting Engaged,Sports And Outdoors,-1673,136.35,51,-32.80,2014-03-20,Return
428,Inactive Merchant,Casual Buyer,Sports And Outdoors,-2089,34.59,68,-30.72,2014-08-07,Return
437,Inactive Merchant,Low Engaged,Home And Garden,-16,2.30,36,-0.44,2014-07-09,Return
552,Unverified Merchant,Guest,Digital Games And Software,-125,0.00,2,-62.50,2014-11-12,Return
...,...,...,...,...,...,...,...,...,...
3710,Inactive Merchant,New,Sports And Outdoors,-5828,-80.49,69,84.46,2017-06-03,Return
3711,Low Volume,New,Other,-8847,-213.65,432,20.48,2015-03-23,Return
3712,Non Merchant,Super Engaged,Automotive,-2839045,-75617.74,58911,48.19,2016-03-08,Return
3713,Medium Volume,Medium Engaged,Digital Games And Software,-111598216,-3829715.17,2217814,50.32,2015-01-06,Return


In [82]:
df_clean

,Merchant_Category,Buyer_Category,Departments,Volume,Revenue,Transaction_Count,Avg_Selling_Price,Date,Volume_Flag
0,Very Low Volume,Low Engaged,Other,570,24.55,281,2.03,2014-11-24,Normal
1,Medium Volume,Super Engaged,Sports And Outdoors,20733354,751838.97,139536,148.59,2014-02-27,Normal
3,Very High Volume,One-And-Done,Clothing And Shoes,142136,4403.07,275,516.86,2014-05-21,Normal
4,Very Low Volume,Guest,Books,839179,30726.21,17538,47.85,2014-05-06,Normal
5,Very High Volume,New,Books,186029680,5415390.11,2184582,85.16,2014-04-25,Normal
...,...,...,...,...,...,...,...,...,...
3313,Non Merchant,New,Books,204748865,7669601.46,3662473,55.90,2017-08-03,Normal
3314,New,Guest,Clothing And Shoes,27580,262.47,164,168.17,2017-10-06,Normal
3354,Inactive Merchant,New,Home And Garden,0,-17.81,53,0.00,2014-07-26,Normal
3383,Inactive Merchant,New,Automotive,0,-0.01,3,-0.13,2016-03-28,Normal


#  Final check: is the data clean?

In [88]:
print('Shape:', df_clean.shape)
print('\nData types:')
print(df_clean.dtypes)
print('\nNull values:')
print(df_clean.isnull().sum())

Shape: (3283, 9)

Data types:
Merchant_Category            object
Buyer_Category               object
Departments                  object
Volume                        int64
Revenue                     float64
Transaction_Count             int64
Avg_Selling_Price           float64
Date                 datetime64[ns]
Volume_Flag                  object
dtype: object

Null values:
Merchant_Category    0
Buyer_Category       0
Departments          0
Volume               0
Revenue              0
Transaction_Count    0
Avg_Selling_Price    0
Date                 0
Volume_Flag          0
dtype: int64


In [90]:
print('Shape:', df_returns.shape)
print('\nData types:')
print(df_returns.dtypes)
print('\nNull values:')
print(df_returns.isnull().sum())

Shape: (432, 9)

Data types:
Merchant_Category            object
Buyer_Category               object
Departments                  object
Volume                        int64
Revenue                     float64
Transaction_Count             int64
Avg_Selling_Price           float64
Date                 datetime64[ns]
Volume_Flag                  object
dtype: object

Null values:
Merchant_Category    0
Buyer_Category       0
Departments          0
Volume               0
Revenue              0
Transaction_Count    0
Avg_Selling_Price    0
Date                 0
Volume_Flag          0
dtype: int64


# Save to CSV (needed for SQL later)

In [93]:
df_clean.to_csv('_Amazon_clean_data.csv', index=False)
df_returns.to_csv('_Amazon_returns_data.csv', index=False)

print('Saved!')
print('  _Amazon_clean_data.csv   → use for EDA and SQL')
print('  _Amazon_returns_data.csv → negative rows kept separately')

Saved!
  _Amazon_clean_data.csv   → use for EDA and SQL
  _Amazon_returns_data.csv → negative rows kept separately


# Push Tables Directly Into MySQL

In [96]:
import pandas as pd
from sqlalchemy import create_engine

MYSQL_USER     = 'root'
MYSQL_PASSWORD = '%40binduvansh8'
MYSQL_HOST     = 'localhost'
MYSQL_PORT     = '3306'
MYSQL_DB       = 'Amazon_database'

engine = create_engine(
    f"mysql+mysqlconnector://{MYSQL_USER}:{MYSQL_PASSWORD}@{MYSQL_HOST}:{MYSQL_PORT}/{MYSQL_DB}"
)

In [98]:
tables = {
    'clean_data'   : df_clean,
    'returns_data' : df_returns,
}

for name, dataframe in tables.items():
    dataframe.to_sql(
        name      = name,
        con       = engine,
        if_exists = 'replace',
        index     = False,
        chunksize = 500
    )
    print(f'Pushed {name:15s} — {len(dataframe):,} rows')

print("\nAll tables sent to MySQL successfully!")

Pushed clean_data      — 3,283 rows
Pushed returns_data    — 432 rows

All tables sent to MySQL successfully!


In [100]:
from sqlalchemy import text

with engine.connect() as conn:
    for t in ['clean_data','returns_data']:
        count = conn.execute(text(f"SELECT COUNT(*) FROM {t}")).fetchone()[0]
        print(f'{t:20s} → {count:,} rows')

clean_data           → 3,283 rows
returns_data         → 432 rows
